# 02 — Data Classification, Quality & Governance

Applies Unity Catalog sensitivity tags, sets up a Lakehouse Monitor for data quality, enforces phone number masking via column masks, and adds AI-ready table and column comments across all Silver tables.

**Tables:** `candidates` (20 candidates, 8 feature scores embedded), `job_requirements` (4 roles)  
**Catalog:** `bx4` &nbsp;|&nbsp; **Schema:** `hrd_2030`

## Setup — Configuration Parameters

Read catalog and schema widget values that parameterize every SQL statement in this notebook.

In [0]:
dbutils.widgets.text("catalog", "bx4",      "Catalog")
dbutils.widgets.text("schema",  "hrd_2030", "Schema")

catalog = dbutils.widgets.get("catalog")
schema  = dbutils.widgets.get("schema")

print("catalog:", catalog)
print("schema: ", schema)

## Section A: Data Classification

Apply Unity Catalog sensitivity tags to PII and ML-critical columns.  
Tags drive downstream policy enforcement, audit, and catalog discovery.

In [ ]:
# Apply sensitivity tags via Unity Catalog ALTER TABLE ... SET TAGS
# Note: tag values are constrained by workspace tag policies.
# Allowed sensitivity values: [pii, internal, public]
# Allowed pii_type values: [ssn, email, phone, credit_card, name, address]
tag_statements = [
    (f"""
    ALTER TABLE {catalog}.{schema}.candidates
      ALTER COLUMN email SET TAGS (
        'sensitivity' = 'internal',
        'pii_type'    = 'email'
      )
    """, "candidates.email"),
    (f"""
    ALTER TABLE {catalog}.{schema}.candidates
      ALTER COLUMN phone SET TAGS (
        'sensitivity' = 'internal',
        'pii_type'    = 'phone'
      )
    """, "candidates.phone"),
    (f"""
    ALTER TABLE {catalog}.{schema}.candidates
      ALTER COLUMN hired SET TAGS (
        'sensitivity' = 'internal'
      )
    """, "candidates.hired"),
]

for stmt, label in tag_statements:
    try:
        spark.sql(stmt)
        print(f"  ✓ Tagged {label}")
    except Exception as e:
        print(f"  WARNING: Could not tag {label}: {e}")

print("\nData classification tags applied.")

## Section B: Data Quality Monitor

Create a snapshot Data Quality Monitor on the `candidates` table for column-level profiling (null rates, distributions, drift metrics).

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import MonitorSnapshot

w = WorkspaceClient()
table_name = f"{catalog}.{schema}.candidates"

try:
    w.quality_monitors.create(
        table_name=table_name,
        assets_dir=f"/Workspace/Shared/hrd_2030_monitoring/candidates",
        output_schema_name=f"{catalog}.{schema}",
        snapshot=MonitorSnapshot(),
    )
    print(f"✅ Data quality monitor created: {table_name}")
    print(f"   profile metrics : {catalog}.{schema}.candidates_profile_metrics")
    print(f"   drift metrics   : {catalog}.{schema}.candidates_drift_metrics")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"ℹ️  Monitor already exists for {table_name} — skipping.")
    else:
        print(f"⚠️  Could not create monitor: {e}")

## Section C: Apply Governance — Phone Number Masking

Create a column masking function and bind it to the `phone` column.  
Members of `jj_hr_digital_admin` see the full phone number; all other users see `***-***-XXXX`.

In [ ]:
# Create column mask function
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.mask_phone(phone STRING)
  RETURN CASE
    WHEN is_account_group_member('jj_hr_digital_admin') THEN phone
    ELSE CONCAT('***-***-', RIGHT(phone, 4))
  END
""")
print(f"Masking function created: {catalog}.{schema}.mask_phone")

# Bind the mask to candidates.phone
spark.sql(f"""
ALTER TABLE {catalog}.{schema}.candidates
  ALTER COLUMN phone SET MASK {catalog}.{schema}.mask_phone
""")
print("Column mask applied to candidates.phone")

In [ ]:
# Verify masking: phone should appear as ***-***-XXXX for non-admin users
result = spark.sql(f"""
  SELECT candidate_id, first_name, last_name, phone
  FROM {catalog}.{schema}.candidates
  LIMIT 5
""")
result.show(truncate=False)
print("\nPhone values are masked for non-admin users (***-***-XXXX format).")
print("Members of 'jj_hr_digital_admin' group will see full phone numbers.")

## Section D: Make AI Ready — Table & Column Comments

Add rich COMMENT ON statements so that Genie, Unity Catalog Search, and AI agents  
can understand the business context of every table and column without guessing.

In [ ]:
# -----------------------------------------------------------------------
# Table-level comments
# -----------------------------------------------------------------------
spark.sql(f"""
COMMENT ON TABLE {catalog}.{schema}.candidates IS
  'Silver-layer HR candidate profile table for Jackson and Jackson HR Digital initiative.
Contains 20 candidates (C001-C020) across 4 open HR roles (JR001-JR004).
C001-C010 have complete feature scores and hired labels (training set).
C011-C020 have partial scores — skills_match_score, interview_score, culture_fit, and hired are NULL
until after their interviews (inference candidates).
Includes 8 scoring dimensions (education, experience, leadership, certification, skills_match,
industry_relevance, interview, culture_fit) and masked PII (phone number).
Primary key: candidate_id. Foreign key: job_id -> job_requirements.job_id.'
""")

spark.sql(f"""
COMMENT ON TABLE {catalog}.{schema}.job_requirements IS
  'Silver-layer job requirements table for Jackson and Jackson HR Digital open HR positions.
Contains 4 active roles: JR001=Director of HR, JR002=VP Talent Acquisition,
JR003=Director of Compensation and Benefits, JR004=Chief People Officer.
Join to candidates on job_id to associate candidates with their target role.
Primary key: job_id.'
""")

print("Table comments added for candidates and job_requirements.")

In [ ]:
# -----------------------------------------------------------------------
# Column comments: candidates
# -----------------------------------------------------------------------
candidate_comments = {
    "candidate_id":              "Primary key. Unique candidate identifier, format C001-C020. C001-C010=training set, C011-C020=new candidates.",
    "first_name":                "Candidate first name. PII — stored in plaintext.",
    "last_name":                 "Candidate last name. PII — stored in plaintext.",
    "email":                     "Candidate professional email address. PII, sensitivity=internal. Used for recruiter outreach.",
    "phone":                     "Candidate phone number. PII, sensitivity=internal. Masked as ***-***-XXXX for non-admins via mask_phone function.",
    "current_title":             "Candidate current job title (e.g., VP of Human Resources, CHRO).",
    "current_company":           "Candidate current employer organization name.",
    "years_total_experience":    "Total years of professional work experience. Integer.",
    "years_hr_experience":       "Years of experience in Human Resources roles specifically. Integer.",
    "years_leadership":          "Years in people-management or director-level leadership roles. Integer.",
    "education_level":           "Highest education degree attained. Enum: BA, MA, MBA, PhD.",
    "certifications":            "Semicolon-separated list of HR certifications held (e.g., SPHR; SHRM-SCP; PHR).",
    "industries":                "Comma-separated list of industries where candidate has worked.",
    "direct_reports_managed":    "Maximum number of direct reports managed. Integer.",
    "budget_managed_millions":   "Largest HR or functional budget managed, in USD millions. Decimal.",
    "location":                  "Candidate current city and state (e.g., New Brunswick NJ).",
    "resume_filename":           "Filename of the candidate PDF resume stored in the Unity Catalog Volume.",
    "job_id":                    "Foreign key to job_requirements.job_id. JR001=Director HR, JR002=VP TA, JR003=Dir Comp&Ben, JR004=CPO.",
    "education_score":           "Education alignment score, 0-100. Reflects degree level and field relevance to the target HR role.",
    "experience_score":          "Total and HR-specific experience score, 0-100. Higher for more years in senior HR roles.",
    "leadership_score":          "Leadership experience score, 0-100. Based on years leading teams, direct reports, and budget.",
    "certification_score":       "HR certification score, 0-100. SPHR or SHRM-SCP=95-100, PHR or SHRM-CP=65-80, none=0.",
    "skills_match_score":        "Skills match score vs. job requirements, 0-100. NULL for C011-C020 until post-interview assessment.",
    "industry_relevance_score":  "Industry relevance score, 0-100. Pharma and healthcare score highest for Jackson and Jackson roles.",
    "interview_score":           "Structured interview performance score, 0-100. NULL for C011-C020 until after the interview.",
    "culture_fit":               "Culture fit rating, 0-100. Based on Jackson and Jackson values alignment and leadership style. NULL for C011-C020.",
    "hired":                     "Binary ML training label. 1=selected for role, 0=not selected. NULL for C011-C020 (pending). sensitivity=internal.",
    "ingested_at":               "Audit timestamp when this row was written to the Silver table.",
}

applied = 0
for col, comment in candidate_comments.items():
    try:
        spark.sql(f"ALTER TABLE {catalog}.{schema}.candidates ALTER COLUMN {col} COMMENT '{comment}'")
        applied += 1
    except Exception as e:
        print(f"  WARNING: Could not comment {col}: {e}")

print(f"Column comments applied to candidates ({applied}/{len(candidate_comments)} columns)")

In [ ]:
# -----------------------------------------------------------------------
# Column comments: job_requirements
# Columns: job_id, title, department, location, min_years_experience,
#   min_hr_years, min_leadership_years, required_education,
#   preferred_certifications, required_skills, preferred_skills,
#   salary_min, salary_max, team_size, reporting_to
# -----------------------------------------------------------------------
job_req_comments = {
    "job_id":                    "Primary key. Unique job identifier. JR001=Director of HR, JR002=VP Talent Acquisition, JR003=Director Comp & Benefits, JR004=Chief People Officer.",
    "title":                     "Official job title being hired for.",
    "department":                "Jackson and Jackson department or business unit posting the role.",
    "location":                  "Job location city and state (e.g., New Brunswick NJ).",
    "min_years_experience":      "Minimum total years of professional experience required. Integer.",
    "min_hr_years":              "Minimum years of HR-specific experience required. Integer.",
    "min_leadership_years":      "Minimum years of leadership or people-management experience required. Integer.",
    "required_education":        "Minimum education level required.",
    "preferred_certifications":  "Semicolon-separated list of preferred HR certifications.",
    "required_skills":           "Comma-separated list of required competencies for this role.",
    "preferred_skills":          "Comma-separated list of preferred/nice-to-have competencies.",
    "salary_min":                "Minimum annual salary offered in USD. Double.",
    "salary_max":                "Maximum annual salary offered in USD. Double.",
    "team_size":                 "Size of the team this role will lead. Integer.",
    "reporting_to":              "Title of the executive this role reports to.",
}

applied = 0
for col, comment in job_req_comments.items():
    try:
        spark.sql(f"ALTER TABLE {catalog}.{schema}.job_requirements ALTER COLUMN {col} COMMENT '{comment}'")
        applied += 1
    except Exception as e:
        print(f"  WARNING: Could not comment {col}: {e}")

print(f"Column comments applied to job_requirements ({applied}/{len(job_req_comments)} columns)")

In [ ]:
# -----------------------------------------------------------------------
# training_data table has been retired — feature scores are now embedded
# in the candidates table. No column comments needed for that table.
# -----------------------------------------------------------------------
print("Note: training_data table is no longer used.")
print("Feature scores (education_score … culture_fit) and hired label are")
print("embedded directly in the candidates Silver table.")
print("Column comments for those columns were applied in the cell above.")

## Section E: Enforce Primary Key Constraints

Declare `NOT NULL` and `PRIMARY KEY` constraints on the Silver tables. Unity Catalog stores these for documentation and lineage; they are not enforced at write time but are visible to Genie and data consumers.

In [0]:
# -----------------------------------------------------------------------
# Primary key constraints
# -----------------------------------------------------------------------
pk_statements = [
    f"ALTER TABLE {catalog}.{schema}.candidates ALTER COLUMN candidate_id SET NOT NULL",
    f"ALTER TABLE {catalog}.{schema}.candidates ADD CONSTRAINT pk_candidates PRIMARY KEY (candidate_id)",
    f"ALTER TABLE {catalog}.{schema}.job_requirements ALTER COLUMN job_id SET NOT NULL",
    f"ALTER TABLE {catalog}.{schema}.job_requirements ADD CONSTRAINT pk_job_requirements PRIMARY KEY (job_id)",
]

for stmt in pk_statements:
    try:
        spark.sql(stmt)
    except Exception as e:
        # Constraint may already exist on re-run
        print(f"Note: {e}")

print("Primary key constraints set:")
print("  candidates.candidate_id -> pk_candidates")
print("  job_requirements.job_id -> pk_job_requirements")

## Verification — Governance Summary

Confirm all governance layers have been applied: tags, monitor, phone mask, column comments, and primary key constraints.

In [ ]:
# -----------------------------------------------------------------------
# Final verification: show enriched metadata on Silver tables
# -----------------------------------------------------------------------
print("=" * 60)
print("Governance summary for hrd_2030 Silver tables")
print("=" * 60)

for table in ["candidates", "job_requirements"]:
    cols = spark.sql(f"DESCRIBE TABLE {catalog}.{schema}.{table}").count()
    print(f"  {table}: {cols} columns described")

print()
print("Completed: Tags, Monitor, Phone Mask, Comments, Primary Keys")